In [1]:
from transformers import pipeline

#### try out models

##### sentiment analysis

> visit `hugging face` website to find different models on different tasks with their documentation

In [2]:
sentiment_classifier = pipeline("sentiment-analysis", model = "cardiffnlp/twitter-roberta-base-sentiment-latest")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
sentiment_classifier("I am finding Large Language Models quite interesting")

[{'label': 'positive', 'score': 0.9572051167488098}]

##### zero-shot classification

In [4]:
zeroshot_classifier = pipeline("zero-shot-classification", model = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [5]:
text = "I would love to explore the world"
labels = ['travel', 'sports', 'geography']

In [6]:
zeroshot_classifier(text, labels)

{'sequence': 'I would love to explore the world',
 'labels': ['travel', 'geography', 'sports'],
 'scores': [0.7553728222846985, 0.2440391629934311, 0.0005880215321667492]}

#### understanding transformer pipeline

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [8]:
sentence = "I can't wait to travel the world."

In [9]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

##### tokenization

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name) # load a pretrained tokenizer specific to that model

In [11]:
input_data = tokenizer(sentence, return_tensors = "pt") # model needs tensors as input; pt for pytorch tensors; tf for tensorflow tensors
print(input_data)

{'input_ids': tensor([[ 101, 1045, 2064, 1005, 1056, 3524, 2000, 3604, 1996, 2088, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


> `input_ids` : numerical id's of tokens based on model's vocabulary

> `token_type_ids` : or segment id's, used by some models to distinguish sentence pairs

> `attention_mask` : indicates which tokens are real (1) vs padded (0), so the model can ignore the padded ones

In [12]:
tokens = tokenizer.convert_ids_to_tokens(input_data['input_ids'][0])
print(tokens)

['[CLS]', 'i', 'can', "'", 't', 'wait', 'to', 'travel', 'the', 'world', '.', '[SEP]']


> AutoTokenizer automatically adds the special tokens required by the specified model

##### get sentiment

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [14]:
with torch.no_grad():
    outputs = model(**input_data) # '**' to unpack dictionary

print(outputs)

SequenceClassifierOutput(loss=None, logits=tensor([[-3.5236,  3.6535]]), hidden_states=None, attentions=None)


In [15]:
softmax = torch.nn.Softmax(dim = 1) # create a softmax layer

probs = softmax(outputs.logits) # pass logits to the layer

In [16]:
pred_class_id = probs.argmax().item() # argmax(): index with max prob, item(): convert tensor to int
print(pred_class_id)

1


In [17]:
label = model.config.id2label[pred_class_id] # model.config: metadata about the model object
print(label)

POSITIVE


##### save models

In [18]:
model_directory = "../models/bert_classifier/"

In [19]:
tokenizer.save_pretrained(model_directory)

('../models/bert_classifier/tokenizer_config.json',
 '../models/bert_classifier/tokenizer.json')

In [20]:
model.save_pretrained(model_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

##### load models

In [21]:
my_tokenizer = AutoTokenizer.from_pretrained(model_directory)

In [22]:
my_model = AutoModelForSequenceClassification.from_pretrained(model_directory)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]